In [8]:
import os
import gc
import time
import numpy as np
import torch

import sys
sys.path.append(".")

from core.design_space import DesignSpace
from core.truth_interface import TruthModelEvaluator
from core.gp_models import GPModelManager
from core.acquisition_functions import AcquisitionFunctionManager
from core.agent_manager import BOAgent, ResourceEvent
from core.logging_utils import LoggingManager
from core.visualization import VisualizationManager

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.double

In [ ]:
# Cell 2: Problem configuration (HEA K/CTE)

# --- Design space: 5D Fe-Co-Cr-Ni-V simplex, matching base_optimizer.py [1][4] ---
N_COMPONENTS = 5
MIN_COMP = 0.10
MAX_COMP = 0.40
STEP = 0.005
SEED = 42

ELEMENT_NAMES = ["Fe", "Co", "Cr", "Ni", "V"]

# Instantiate n-d simplex design space
design_space = DesignSpace(
    components=N_COMPONENTS,
    min_comp=MIN_COMP,
    max_comp=MAX_COMP,
    step=STEP,
    seed=SEED,
    element_names=ELEMENT_NAMES,
)

bounds_np = design_space.get_bounds()  # shape (2, 5)
bounds_t = torch.tensor(bounds_np, dtype=DTYPE, device=DEVICE)

# --- Truth model paths (RF + scalers as in the original interface) [8] ---
MODEL_PATH = "models/RFR_best_model.pkl"
X_SCALER_PATH = "models/x_scaler.pkl"
Y_SCALER_PATH = "models/y_scaler.pkl"

# postprocess mapping: reproduce original TruthModelEvaluator behavior [8]
def postprocess_outputs(preds_raw: np.ndarray) -> np.ndarray:
    """
    Original convention:
      - RF output preds_raw[:,0] = K (W/mK)
      - RF output preds_raw[:,1] = signed CTE_micro
    Old code did: cte = abs(-preds[:,1])
    Here we expose Y[:,0] = CTE, Y[:,1] = K to the BO loop.
    """
    k = preds_raw[:, 0]
    cte_micro = -1.0 * preds_raw[:, 1]
    cte = np.abs(cte_micro)
    return np.column_stack([cte, k])

# Instantiate truth evaluator
evaluator = TruthModelEvaluator(
    model_path=MODEL_PATH,
    x_scaler_path=X_SCALER_PATH,
    y_scaler_path=Y_SCALER_PATH,
    postprocess_outputs=postprocess_outputs,
)

# --- Objective naming and score function (K/|CTE|) [1][6] ---
OBJ1_NAME = "CTE"
OBJ2_NAME = "K"

def score_fn(Y: np.ndarray) -> np.ndarray:
    """
    Scalar progress metric: maximize K / |CTE|.
    Y[:,0] = CTE, Y[:,1] = K.
    """
    cte = Y[:, 0]
    k = Y[:, 1]
    return k / (np.abs(cte) + 1e-12)

SCORE_NAME = "K / |CTE|"

# --- BO resource and loop config (match base_optimizer.py) [1] ---
INIT_N = 10        # initial random samples
ITERS = 10         # max BO iterations

TOTAL_BUDGET = 10000.0   # $
TOTAL_TIME = 20.0        # weeks
COST_PER_POINT = 100.0   # $ per experiment
TIME_PER_ITERATION = 1.0 # weeks per iteration

MC_SAMPLES = 128
POOL_SUBSAMPLE = None    # None = use full enumerated space [1]
TOTAL_BATCH_SIZE = 5     # total points per iteration

# --- LLM / agent config ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
AGENT_MODEL = "gpt-4o"
MAX_REASONING_STEPS = 10

PROBLEM_DESCRIPTION = """
High Entropy Alloy (HEA) optimization in the Fe-Co-Cr-Ni-V composition space.

Objectives:
- Minimize CTE (Coefficient of Thermal Expansion).
- Maximize K (Thermal Conductivity).

Input space: 5D compositions (Fe, Co, Cr, Ni, V), each in [0.1, 0.4], summing to 1.0.
"""

# --- Logging & visualization config ---
LOG_DIR = "results"
EXPERIMENT_NAME = "hea_k_over_cte_simplex"
CREATE_VIS = True   # will use pentagon plot (dim=5)
CREATE_GIF = True

TypeError: DesignSpace.__init__() got an unexpected keyword argument 'n_components'. Did you mean 'components'?

In [3]:
# Cell 3: Instantiate GP manager, acquisition manager, agent, logger, visualization

# GP manager for 2 objectives
gp_manager = GPModelManager(bounds=bounds_t, use_priors=False)  # priors optional [5]

# Acquisition manager: 5D simplex, batch size 5, equality constraints for simplex [2]
from core.acquisition_functions import AcquisitionFunctionManager

acq_manager = AcquisitionFunctionManager(
    bounds=bounds_t,
    total_batch_size=TOTAL_BATCH_SIZE,
    equality_constraints=None,  # we'll set simplex constraint right after
)
# For 5D simplex, use its simplex equality helper
simplex_constraints = acq_manager._get_simplex_equality_tuple()
acq_manager.equality_constraints = simplex_constraints

# Agent
agent_log_dir = os.path.join(LOG_DIR, EXPERIMENT_NAME, "agent_logs")
os.makedirs(agent_log_dir, exist_ok=True)

agent = BOAgent(
    api_key=OPENAI_API_KEY,
    model=AGENT_MODEL,
    log_dir=agent_log_dir,
    max_reasoning_steps=MAX_REASONING_STEPS,
    temperature=0.7,
    problem_description=PROBLEM_DESCRIPTION,
    obj1_name=OBJ1_NAME,
    obj2_name=OBJ2_NAME,
)

# Optional: schedule resource events
events: list[ResourceEvent] = []
# Example:
# events.append(ResourceEvent(iteration=5, event_type="budget_change",
#                             description="Budget cut by 50%", modifier=lambda x: x * 0.5))
for ev in events:
    agent.add_event(ev)

# Logging manager
exp_dir = os.path.join(LOG_DIR, EXPERIMENT_NAME)
os.makedirs(exp_dir, exist_ok=True)
logger = LoggingManager(exp_dir, EXPERIMENT_NAME)

# Visualization manager (pentagon plot valid since dim=5 and design_space.space exists) [9]
vis_manager = VisualizationManager(exp_dir, design_space, experiment_name=EXPERIMENT_NAME) if CREATE_VIS else None

[Logger] Initialized logging in results\hea_k_over_cte_simplex
[Logger] Files:
  - Convergence: results\hea_k_over_cte_simplex\hea_k_over_cte_simplex_convergence.csv
  - Options: results\hea_k_over_cte_simplex\hea_k_over_cte_simplex_options.csv
  - Evaluations: results\hea_k_over_cte_simplex\hea_k_over_cte_simplex_evaluations.csv
[Viz] Subsampled design space from 3804225 to 100000
[Viz] Setup affine projection: 100000 points
[Viz] Initialized pentagon visualization in results\hea_k_over_cte_simplex\visualizations
[Viz] Experiment name: hea_k_over_cte_simplex


In [4]:
# Cell 4: Agent configures GP kernel (optional)

print("\n" + "="*80)
print("[NB] Agent configuring GP kernel...")
print("="*80)

kernel_code, kernel_reasoning = agent.configure_gp_kernel(use_priors=False)

if kernel_code:
    print("[NB] Agent requested custom kernel (not implemented in this notebook). Using default kernel.")
else:
    print("[NB] Agent selected default kernel.")


[NB] Agent configuring GP kernel...

[Agent] Configuring GP Kernel

[Agent] Reasoning step 1/10...

REASONING:

1. **Kernel Properties**: For the input space consisting of compositions that sum to 1, it is essential to capture the relationships between these constrained variables. The kernel should be able to model correlations between inputs effectively, especially considering the compositional nature of the data.

2. **Default Kernel Sufficiency**: The RBF (Radial Basis Function) or Matern kernels are widely used due to their flexibility and smoothness properties. They can model a wide range of functions and are typically a good starting point. However, they might not inherently capture the simplex constraint (sum to 1) of the input space.

3. **Custom Kernel Considerations**: A custom kernel, such as a Dirichlet kernel or a kernel specifically designed for compositional data, could potentially capture the simplex constraint more naturally. However, designing such a kernel requires 

In [6]:
# Cell 5: Initial random sampling and evaluation

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"\n[NB] Initializing with {INIT_N} random samples from simplex design space")
X0_np = design_space.sample(INIT_N, method="random")  # (INIT_N, 5) [4]
result0 = evaluator.evaluate(X0_np)
Y0_np = result0.y  # shape (INIT_N, 2) = [CTE, K]

X = torch.tensor(X0_np, dtype=DTYPE, device=DEVICE)
Y = torch.tensor(Y0_np, dtype=DTYPE, device=DEVICE)

# Resource accounting: initial sampling consumes 1 iteration's worth of time [1]
initial_budget_spent = INIT_N * COST_PER_POINT
initial_time_spent = TIME_PER_ITERATION
budget_remaining = TOTAL_BUDGET - initial_budget_spent
time_remaining = TOTAL_TIME - initial_time_spent

print(f"[NB] Initial sampling used: ${initial_budget_spent:.2f} / {initial_time_spent:.1f} weeks")
print(f"[NB] Remaining resources: ${budget_remaining:.2f} / {time_remaining:.1f} weeks")

# Log initial state
logger.log_iteration(
    iteration=0,
    X_history=X.cpu().numpy(),
    Y_history=Y.cpu().numpy(),
    n_new_points=INIT_N,
    strategy=agent,
    timing=0.0,
    extra_info={"note": "initial_sampling", "agent_kernel_reasoning": kernel_reasoning},
    acquisition_data=None,
    score_fn=score_fn,
    obj1_name=OBJ1_NAME,
    obj2_name=OBJ2_NAME,
)
logger.log_evaluations(
    iteration=0,
    X_new=X0_np,
    Y_new=Y0_np,
    selection_info=None,
    score_fn=score_fn,
    obj1_name=OBJ1_NAME,
    obj2_name=OBJ2_NAME,
)



[NB] Initializing with 10 random samples from simplex design space
[NB] Initial sampling used: $1000.00 / 1.0 weeks
[NB] Remaining resources: $9000.00 / 19.0 weeks
[Logger] Logged convergence for iteration 0
[Logger] Logged 10 evaluations for iteration 0


c:\Users\trippr5118\.conda\envs\tc-python\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [7]:
# Cell 6: Main Bayesian Optimization loop

iteration = 0

while iteration < ITERS:
    # Resource check: need at least enough for one point and one iteration
    if budget_remaining < COST_PER_POINT or time_remaining < TIME_PER_ITERATION:
        print(f"\n[NB] Stopping: insufficient budget (${budget_remaining:.2f}) or time ({time_remaining:.1f} weeks)")
        break

    iteration += 1
    iter_start = time.perf_counter()

    # Apply any scheduled events for this iteration
    budget_remaining, time_remaining, COST_PER_POINT, event_msgs = agent.process_iteration_events(
        iteration, budget_remaining, time_remaining, COST_PER_POINT
    )
    if event_msgs:
        print("\n" + "="*80)
        for msg in event_msgs:
            print(msg)
        print("="*80)

    print("\n" + "="*80)
    print(f"[NB] Iteration {iteration}/{ITERS}")
    print(f"[NB] Budget: ${budget_remaining:.2f} | Time: {time_remaining:.1f} weeks")
    print("="*80)

    # Fit GP model
    print("[NB] Fitting MultiTaskGP model...")
    model = gp_manager.fit_model(X, Y)  # Y[:,0]=CTE, Y[:,1]=K [5]

    # Compute Pareto front in (CTE, K) space. For acquisition, we want both objectives maximized,
    # so you can transform CTE to [-CTE] before calling compute_pareto_front, or keep as-is and
    # interpret appropriately. Here we keep Y as-is and assume AcquisitionFunctionManager expects
    # "both to be maximized", so we pass (-CTE, K).
    Y_np = Y.cpu().numpy()
    Y_for_pareto = torch.tensor(
        np.column_stack([-Y_np[:, 0], Y_np[:, 1]]),
        dtype=DTYPE,
        device=DEVICE,
    )
    pareto_Y, ref_point = acq_manager.compute_pareto_front(Y_for_pareto)

    # Candidate pool: full enumerated simplex if POOL_SUBSAMPLE is None [1][4]
    if POOL_SUBSAMPLE is None:
        X_pool_np = design_space.space
    else:
        X_pool_np = design_space.sample(POOL_SUBSAMPLE, method="sobol")
    X_pool = torch.tensor(X_pool_np, dtype=DTYPE, device=DEVICE)

    # Compute acquisition allocation options (0–5 exploit points) [2]
    print("[NB] Computing acquisition options...")
    allocation_options = acq_manager.compute_all_allocation_options(
        model=model,
        X_pool=X_pool,
        pareto_Y=pareto_Y,
        ref_point=ref_point,
        mc_samples=MC_SAMPLES,
    )

    # Agent selects allocation [3]
    print("[NB] Consulting LLM agent for resource allocation...")
    selected_idx, reasoning, selected_point_arrays = agent.select_resource_allocation(
        iteration=iteration,
        budget_remaining=budget_remaining,
        time_remaining=time_remaining,
        cost_per_point=COST_PER_POINT,
        time_per_point=TIME_PER_ITERATION,
        X_history=X.cpu().numpy(),
        Y_history=Y.cpu().numpy(),
        allocation_options=allocation_options,
        max_batch_size=TOTAL_BATCH_SIZE,
        score_fn=score_fn,
    )

    # Combine selected points
    if selected_point_arrays:
        X_next_np = np.vstack(selected_point_arrays)
    else:
        print("[NB] Agent selected no points; using balanced option (index 2) as fallback.")
        batch = allocation_options.qehvi_batches[2]  # 2 explo / 3 explore
        parts = []
        if batch.points.size > 0:
            parts.append(batch.points)
        if batch.explore_points.size > 0:
            parts.append(batch.explore_points)
        X_next_np = np.vstack(parts) if parts else design_space.sample(1, method="random")

    print(f"[NB] Evaluating {len(X_next_np)} selected points...")
    result_next = evaluator.evaluate(X_next_np)
    Y_next_np = result_next.y  # (k, 2)

    # Update datasets
    X = torch.cat([X, torch.tensor(X_next_np, dtype=DTYPE, device=DEVICE)], dim=0)
    Y = torch.cat([Y, torch.tensor(Y_next_np, dtype=DTYPE, device=DEVICE)], dim=0)

    # Resource updates (each iteration consumes TIME_PER_ITERATION weeks regardless of batch) [1]
    points_used = len(X_next_np)
    budget_spent = points_used * COST_PER_POINT
    time_spent = TIME_PER_ITERATION
    budget_remaining -= budget_spent
    time_remaining -= time_spent

    iter_time = time.perf_counter() - iter_start

    # Logging
    extra_info = {
        "agent_selected_option": selected_idx,
        "agent_reasoning": reasoning[:200] + "..." if len(reasoning) > 200 else reasoning,
        "budget_remaining": budget_remaining,
        "time_remaining": time_remaining,
        "budget_spent_this_iter": budget_spent,
        "time_spent_this_iter": time_spent,
    }

    logger.log_iteration(
        iteration=iteration,
        X_history=X.cpu().numpy(),
        Y_history=Y.cpu().numpy(),
        n_new_points=points_used,
        strategy=agent,
        timing=iter_time,
        extra_info=extra_info,
        acquisition_data=allocation_options,
        score_fn=score_fn,
        obj1_name=OBJ1_NAME,
        obj2_name=OBJ2_NAME,
    )
    logger.log_evaluations(
        iteration=iteration,
        X_new=X_next_np,
        Y_new=Y_next_np,
        selection_info=None,
        score_fn=score_fn,
        obj1_name=OBJ1_NAME,
        obj2_name=OBJ2_NAME,
    )

    # Visualization (pentagon plot) [9]
    if vis_manager is not None:
        vis_manager.create_iteration_plot(
            iteration=iteration,
            X_history=X.cpu().numpy(),
            Y_history=Y.cpu().numpy(),
            X_new=X_next_np,
            model=model,
            bounds=bounds_t,
            selection_info={"agent_option": selected_idx},
            score_fn=score_fn,
            score_name=SCORE_NAME,
            obj1_name=OBJ1_NAME,
            obj2_name=OBJ2_NAME,
        )

    # Print iteration summary
    scores_all = score_fn(Y.cpu().numpy())
    best_score = float(np.max(scores_all))
    print(f"\n[NB] Iteration {iteration} complete in {iter_time:.2f}s")
    print(f"[NB] Best {SCORE_NAME}: {best_score:.3f}")
    print(f"[NB] Remaining resources: ${budget_remaining:.2f} / {time_remaining:.1f} weeks")

    # Cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "="*80)
print("[NB] Optimization complete.")
print("="*80)


[NB] Iteration 1/10
[NB] Budget: $9000.00 | Time: 19.0 weeks
[NB] Fitting MultiTaskGP model...


c:\Users\trippr5118\.conda\envs\tc-python\Lib\site-packages\linear_operator\utils\interpolation.py:71: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:80.)
  summing_matrix = cls(summing_matrix_indices, summing_matrix_values, size)
c:\Users\trippr5118\.conda\envs\tc-python\Lib\site-packages\linear_operator\utils\interpolation.py:71: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:655.)
  summing_matrix = cls(summing_matrix_indices, summing_matrix_values, size)


[NB] Computing acquisition options...
[AcqFn] Computing acquisition functions.
[AcqFn] Using 128 samples for qEHVI, 64 for MO-MESMO
[AcqFn] Computing top entropy reference points...

[AcqFn] Option 0: 5 exploit + 0 explore
  Optimizing qEHVI with q=5...
  → Entropy: 21.2747, QEHVI: 6.0508

[AcqFn] Option 1: 4 exploit + 1 explore
  Optimizing qEHVI with q=4...
  Optimizing MO-MESMO with q=1...
  ✓ MO-MESMO succeeded
  → Entropy: 17.6515, QEHVI: 6.0384

[AcqFn] Option 2: 3 exploit + 2 explore
  Optimizing qEHVI with q=3...
  Optimizing MO-MESMO with q=2...
  ✓ MO-MESMO succeeded
  → Entropy: 13.3300, QEHVI: 6.0163

[AcqFn] Option 3: 2 exploit + 3 explore
  Optimizing qEHVI with q=2...
  Optimizing MO-MESMO with q=3...
  ✓ MO-MESMO succeeded
  → Entropy: 10.3752, QEHVI: 5.9787

[AcqFn] Option 4: 1 exploit + 4 explore
  Optimizing qEHVI with q=1...
  Optimizing MO-MESMO with q=4...
  ✓ MO-MESMO succeeded
  → Entropy: 5.7589, QEHVI: 5.9202

[AcqFn] Option 5: 0 exploit + 5 explore
  Optimizi

c:\Users\trippr5118\Documents\resource-allocation\core\agent_manager.py:170: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  return ConversationBufferMemory(memory_key="chat_history", return_messages=True)



1. **Runway and Budget Consideration**: We have a budget for approximately 90 more data points with 19 iterations remaining, indicating we can afford about 5 points per iteration. This aligns well with any of the current options.

2. **Progress Momentum**: The progress is steady with a slight increase of 0.6% in the objective score over the last three iterations, suggesting that our current model is improving gradually.

3. **Exploit vs. Explore**: The best observed point has a CTE of 12.77 and K of 10.15 with a balanced composition. Given the steady progress, this suggests that there is some potential for improvement by exploiting known good regions.

4. **EHVI and Entropy Analysis**: 
   - Option 0 offers the highest EHVI (6.0508) and entropy (21.27), indicating it maximizes expected improvement and has higher uncertainty reduction, suggesting that there might be a high potential for improving the current best point.
   - Option 1 presents a slight reduction in EHVI and entropy, ind

...

[Viz] Predicted scores - Min: -0.0813, Max: 6.9715, Mean: 0.7032
[Viz] Created plot for iteration 8

[NB] Iteration 8 complete in 61.04s
[NB] Best K / |CTE|: 7.157
[NB] Remaining resources: $5000.00 / 11.0 weeks

[NB] Iteration 9/10
[NB] Budget: $5000.00 | Time: 11.0 weeks
[NB] Fitting MultiTaskGP model...
[NB] Computing acquisition options...
[AcqFn] Computing acquisition functions.
[AcqFn] Using 128 samples for qEHVI, 64 for MO-MESMO
[AcqFn] Computing top entropy reference points...

[AcqFn] Option 0: 5 exploit + 0 explore
  Optimizing qEHVI with q=5...
  → Entropy: 13.2483, QEHVI: 7.5523

[AcqFn] Option 1: 4 exploit + 1 explore
  Optimizing qEHVI with q=4...
  Optimizing MO-MESMO with q=1...
  ✓ MO-MESMO succeeded
  → Entropy: 16.1820, QEHVI: 7.5093

[AcqFn] Option 2: 3 exploit + 2 explore
  Optimizing qEHVI with q=3...
  Optimizing MO-MESMO with q=2...
  ✓ MO-MESMO succeeded
  → Entropy: 14.4685, QEHVI: 7.5092

[AcqFn] Option 3: 2 exploit + 3 explore
  Optimizing qEHVI with q=2...


c:\Users\trippr5118\.conda\envs\tc-python\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[Viz] Predicted scores - Min: -0.0884, Max: 7.6725, Mean: 0.7911
[Viz] Created plot for iteration 9

[NB] Iteration 9 complete in 68.81s
[NB] Best K / |CTE|: 7.157
[NB] Remaining resources: $4500.00 / 10.0 weeks

[NB] Iteration 10/10
[NB] Budget: $4500.00 | Time: 10.0 weeks
[NB] Fitting MultiTaskGP model...
[NB] Computing acquisition options...
[AcqFn] Computing acquisition functions.
[AcqFn] Using 128 samples for qEHVI, 64 for MO-MESMO
[AcqFn] Computing top entropy reference points...

[AcqFn] Option 0: 5 exploit + 0 explore
  Optimizing qEHVI with q=5...
  → Entropy: 13.6856, QEHVI: 7.4911

[AcqFn] Option 1: 4 exploit + 1 explore
  Optimizing qEHVI with q=4...
  Optimizing MO-MESMO with q=1...
  ✓ MO-MESMO succeeded
  → Entropy: 12.9869, QEHVI: 7.4912

[AcqFn] Option 2: 3 exploit + 2 explore
  Optimizing qEHVI with q=3...
  Optimizing MO-MESMO with q=2...
  ✓ MO-MESMO succeeded
  → Entropy: 12.0264, QEHVI: 7.5286

[AcqFn] Option 3: 2 exploit + 3 explore
  Optimizing qEHVI with q=2...

c:\Users\trippr5118\.conda\envs\tc-python\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[Viz] Predicted scores - Min: -0.4531, Max: 6.8056, Mean: 0.6879
[Viz] Created plot for iteration 10

[NB] Iteration 10 complete in 93.96s
[NB] Best K / |CTE|: 7.157
[NB] Remaining resources: $4000.00 / 9.0 weeks

[NB] Optimization complete.


In [8]:
# Cell 7: Finalization, summary, and GIF

# Save agent memory
agent.save_global_memory()
print(f"[NB] Agent memory saved in {agent_log_dir}")

# Finalize logger
logger.finalize()

# Create GIF of iteration plots
if vis_manager is not None and CREATE_GIF:
    vis_manager.create_gif(duration=2.0, gif_name=f"{EXPERIMENT_NAME}.gif")

# Load and show summary statistics
summary = logger.get_summary_stats(obj1_name=OBJ1_NAME, obj2_name=OBJ2_NAME)
print("\n[NB] Summary statistics:")
for k, v in summary.items():
    print(f"  {k}: {v}")

[NB] Agent memory saved in results\hea_k_over_cte_simplex\agent_logs
[Logger] Finalized logging for hea_k_over_cte_simplex
[Viz] Determining minimum image dimensions for GIF...
[Viz] Created GIF at results\hea_k_over_cte_simplex\hea_k_over_cte_simplex.gif

[NB] Summary statistics:
  total_iterations: 10
  total_evaluations: 60
  best_score: 7.156527490646732
  best_CTE: 10.164449197419358
  best_K: 72.74216010862091
  final_mean_score: 2.180146561212156
  total_time_sec: 663.359856700059
